# Tutorial 08: Statistical Tests and NLP Basics

Author: Georg Ahnert

In this notebook we will do a quick recap of numpy, some basic statistical analyses, and basics of text classification with natural language processing (NLP).

In [11]:
# Install dependencies
!pip install numpy scipy statsmodels nltk scikit-learn

## Numpy Recap

In [36]:
import numpy as np

In [41]:
# initializing arrays
x = np.array([True, 'hello world', 42], dtype = np.float64)  # fails because 'hello world' cannot be converted to np.float64
x

ValueError: could not convert string to float: 'hello world'

In [42]:
# running example
x = np.array([[1,2,3,4], [5,6,7,8], [9,10,11,12], [13,14,15,16]])
x

array([[ 1,  2,  3,  4],
       [ 5,  6,  7,  8],
       [ 9, 10, 11, 12],
       [13, 14, 15, 16]])

In [43]:
x[3, 1]  # 4th row, 2nd column – indices start at 0

14

In [44]:
x[-2,-1]  # second-to-last row, last column

12

In [45]:
x[1, 1:3]  # 2nd row, 2nd-3rd column – use ':' for slicing

array([6, 7])

In [46]:
x[1, :100]  # slicing out of range is fine

array([5, 6, 7, 8])

In [47]:
x[1::2,1::2]  # get all odd indices

array([[ 6,  8],
       [14, 16]])

In [48]:
x[x % 3 == 0]  # all multiples of 3

array([ 3,  6,  9, 12, 15])

In [49]:
x[x % 3 == 0] = 42  # replace all multiples of 3
x

array([[ 1,  2, 42,  4],
       [ 5, 42,  7,  8],
       [42, 10, 11, 42],
       [13, 14, 42, 16]])

In [50]:
x = np.array([[0,3,6,9]]).T  # create a 2D array and transpose it
x

array([[0],
       [3],
       [6],
       [9]])

In [51]:
y = np.array([1,2])  # create a (smaller) 1D array
x*y  # this uses broadcasting

array([[ 0,  0],
       [ 3,  6],
       [ 6, 12],
       [ 9, 18]])

In [52]:
x = np.arange(10)  # works like Python's own range() – see lecture 2
x

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [53]:
x = np.linspace(0, 1, num = 11)  # creates evenly spaced numbers over a specified interval [start, stop]
x

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

## Statistical Analyses

SciPy provides a variety of tools for statistical analyses, including statistical tests, see: https://docs.scipy.org/doc/scipy/tutorial/stats.html

In [55]:
import pandas as pd
import scipy.stats as stats

In [56]:
# We'll use the quality of goverance dataset again as an example

df = pd.read_csv("http://www.qogdata.pol.gu.se/data/qog_std_cs_jan22.csv")

df = df[["cname", "wdi_pop", "wdi_popgr", "wdi_gdpcapcur", "wdi_gdpcapgr", "wdi_area", "wdi_broadb", "ht_region"]]
df = df.rename(columns={"cname" : "name",
            "wdi_pop" : "population",
            "wdi_popgr" : "population_growth",
            "wdi_gdpcapcur" : "gdp_per_capita",
            "wdi_gdpcapgr" : "gdp_growth",
            "wdi_area" : "area",
            "wdi_broadb" : "internet",
            "ht_region" : "region"
           })

regions = {
    1: "Eastern Europe",
    2: "Latin America",
    3: "North Africa & Middle East",
    4: "Sub-Saharan Africa",
    5: "W. Europe & N. America",
    6: "East Asia",
    7: "South-East Asia",
    8: "South Asia",
    9: "Pacific",
    10: "Caribbean",
}

df['region'] = df['region'].replace(regions).astype('category')
df.head()

,name,population,population_growth,gdp_per_capita,gdp_growth,area,internet,region
0,Afghanistan,37171920.0,2.384309,493.756592,-1.194900,652860.0,0.043041,South Asia
1,Albania,2866376.0,-0.246732,5284.380371,4.328395,27400.0,12.555659,Eastern Europe
2,Algeria,42228416.0,2.007399,4153.956055,-0.811233,2381741.0,7.262936,North Africa & Middle East
3,Andorra,77008.0,0.014285,41791.968750,1.574254,470.0,46.311977,W. Europe & N. America
4,Angola,30809788.0,3.276145,3289.644043,-5.162112,1246700.0,0.355605,Sub-Saharan Africa


### Inter-Rater Agreement

In [91]:
from sklearn.metrics import cohen_kappa_score

regions1 = df['region']
regions2 = df['region'].sample(frac=1) # fully shuffled

regions3 = pd.concat([regions2[:50], regions1[50:]]) # shuffle 50 instances

print(cohen_kappa_score(regions1, regions2))
print(cohen_kappa_score(regions1, regions3))

0.005681116462887603
0.7421010325851728


### Is there a significant difference in internet availability between Eastern Europe and East Asia?

In [5]:
data_eastEurope = df[df.region == 'Eastern Europe'].internet
data_eastAsia = df[df.region == 'East Asia'].internet

stats.ttest_ind(data_eastEurope, data_eastAsia, nan_policy='omit') # omitting missing values will likely introduce bias!

TtestResult(statistic=-1.4124966414145717, pvalue=0.1680943007692956, df=30.0)

### What about all pairs of regions?

In [6]:
results = []

for region1 in regions.values():
    for region2 in regions.values():
        test_result = stats.ttest_ind(
            df[df.region == region1].internet,
            df[df.region == region2].internet,
            nan_policy='omit'
        )
        results.append(
            {
                'region 1': region1,
                'region 2': region2,
                'statistic': test_result.statistic,
                'pvalue': test_result.pvalue,
            })

test_df = pd.DataFrame(results)
test_df

,region 1,region 2,statistic,pvalue
0,Eastern Europe,Eastern Europe,0.000000,1.000000e+00
1,Eastern Europe,Latin America,4.214725,1.155980e-04
2,Eastern Europe,North Africa & Middle East,3.113522,3.176735e-03
3,Eastern Europe,Sub-Saharan Africa,12.105036,3.984334e-19
4,Eastern Europe,W. Europe & N. America,-8.393796,2.642705e-11
...,...,...,...,...
95,Caribbean,East Asia,-1.078297,2.979367e-01
96,Caribbean,South-East Asia,2.754597,1.156577e-02
97,Caribbean,South Asia,3.452054,2.670469e-03
98,Caribbean,Pacific,4.440719,2.055847e-04


### How many significant differences can we identify?

In [7]:
test_df['significant'] = test_df.pvalue < 0.01

test_df.significant.value_counts()

significant
True     58
False    42
Name: count, dtype: int64

In [8]:
test_df['corr_pvalue_bh'] = stats.false_discovery_control(test_df.pvalue, method='bh') # apply the Benjamini-Hochberg correction method
(test_df.corr_pvalue_bh < 0.01).value_counts()

corr_pvalue_bh
True     54
False    46
Name: count, dtype: int64

In [9]:
from statsmodels.stats.multitest import multipletests

test_df['corr_pvalue_bonferroni'] = multipletests(test_df.pvalue, method='bonferroni')[1] # Bonferroni correction is more established, but very conservative
(test_df.corr_pvalue_bonferroni < 0.01).value_counts()

corr_pvalue_bonferroni
False    68
True     32
Name: count, dtype: int64

### OLS Regression: How strongly is internet availability influenced by region, gdp_per_capita, and population_growth?

In [33]:
import statsmodels.formula.api as smf

# OLS models can be relatively easily be implemented with the R-style formula API
# categorical columns are automatically one-hot encoded
# "*" in the formula adds interactions between the respective variables
model = smf.ols(
    formula="internet ~ region * gdp_per_capita * population_growth",
    data = df
).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               internet   R-squared:                       0.894
Model:                            OLS   Adj. R-squared:                  0.866
Method:                 Least Squares   F-statistic:                     31.70
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           7.40e-54
Time:                        11:03:56   Log-Likelihood:                -553.17
No. Observations:                 186   AIC:                             1186.
Df Residuals:                     146   BIC:                             1315.
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
=========================================================================================================================================
                                                                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                 2.8241      6.256      0.451      0.652      -9.541      15.189
region[T.East Asia]                                                      27.1977     12.201      2.229      0.027       3.085      51.311
region[T.Eastern Europe]                                                 10.4619      6.563      1.594      0.113      -2.509      23.433
region[T.Latin America]                                                 -11.1929      9.774     -1.145      0.254     -30.509       8.123
region[T.North Africa & Middle East]                                     -6.8900      8.868     -0.777      0.438     -24.416      10.636
region[T.Pacific]                                                        -1.2135      9.001     -0.135      0.893     -19.002      16.575
region[T.South Asia]                                                      0.3876     15.315      0.025      0.980     -29.879      30.655
region[T.South-East Asia]                                                 7.4164      8.125      0.913      0.363      -8.642      23.475
region[T.Sub-Saharan Africa]                                             -5.5804      7.160     -0.779      0.437     -19.730       8.570
region[T.W. Europe & N. America]                                         30.2536      6.710      4.509      0.000      16.993      43.514
gdp_per_capita                                                            0.0021      0.001      3.696      0.000       0.001       0.003
region[T.East Asia]:gdp_per_capita                                       -0.0019      0.001     -2.991      0.003      -0.003      -0.001
region[T.Eastern Europe]:gdp_per_capita                                  -0.0013      0.001     -2.267      0.025      -0.002      -0.000
region[T.Latin America]:gdp_per_capita                                -5.833e-05      0.001     -0.067      0.947      -0.002       0.002
region[T.North Africa & Middle East]:gdp_per_capita                      -0.0009      0.001     -1.370      0.173      -0.002       0.000
region[T.Pacific]:gdp_per_capita                                         -0.0023      0.002     -1.298      0.196      -0.006       0.001
region[T.South Asia]:gdp_per_capita                                      -0.0013      0.004     -0.336      0.737      -0.009       0.006
region[T.South-East Asia]:gdp_per_capita                                 -0.0018      0.001     -2.986      0.003      -0.003      -0.001
region[T.Sub-Saharan Africa]:gdp_per_capita                              -0.0001      0.001     -0.182      0.856      -0.002       0.001
region[T.W. Europe & N. America]:gdp_per_capita

In [34]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# check for multicollinearity by calculating the variance inflaction factors
vif = [variance_inflation_factor(model.model.exog, i)
       for i in range(model.model.exog.shape[1])]
vif

[254.83838121223465,
 20.393453809899913,
 35.8619524154263,
 57.03952098844729,
 46.9556071560864,
 29.347764333476615,
 62.85020974977105,
 23.915453264262624,
 62.123275755048375,
 36.36890156000483,
 1445.8307247956197,
 39.62994507924501,
 47.765701507223774,
 41.649174263898956,
 160.3447048684538,
 41.818213644947015,
 76.56427073989249,
 70.7733121679342,
 13.68665155118842,
 1497.957920853037,
 301.01460850281944,
 11.057992540630655,
 35.71410694449647,
 67.84264054047648,
 134.34586781645905,
 79.90392381881448,
 87.65137950620823,
 28.83641276866906,
 307.6035363847816,
 52.213882187096026,
 2313.66065117148,
 6.518405347679252,
 19.662519241002464,
 47.53062066127477,
 667.1932716092815,
 81.20279237678342,
 108.20929405445845,
 36.774007682524285,
 37.4872413638781,
 1900.1559967321941]

In [35]:
# To mitigate multicollinearity, we can remove the interactions
# Also, let's set "Eastern Europe" as the reference for the region variable

model = smf.ols(
    formula="internet ~ C(region, Treatment(reference='Eastern Europe')) + gdp_per_capita + population_growth",
    data = df
).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               internet   R-squared:                       0.801
Model:                            OLS   Adj. R-squared:                  0.788
Method:                 Least Squares   F-statistic:                     63.49
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           5.32e-55
Time:                        11:04:06   Log-Likelihood:                -612.29
No. Observations:                 186   AIC:                             1249.
Df Residuals:                     174   BIC:                             1287.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
==================================================================================================================================================
                                                                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                         19.3307      1.307     14.786      0.000      16.750      21.911
C(region, Treatment(reference='Eastern Europe'))[T.Caribbean]                      0.5346      2.285      0.234      0.815      -3.975       5.044
C(region, Treatment(reference='Eastern Europe'))[T.East Asia]                      7.1346      3.625      1.968      0.051      -0.021      14.290
C(region, Treatment(reference='Eastern Europe'))[T.Latin America]                 -7.1311      2.106     -3.386      0.001     -11.288      -2.974
C(region, Treatment(reference='Eastern Europe'))[T.North Africa & Middle East]    -4.5658      2.300     -1.985      0.049      -9.105      -0.027
C(region, Treatment(reference='Eastern Europe'))[T.Pacific]                      -14.2357      2.507     -5.679      0.000     -19.183      -9.289
C(region, Treatment(reference='Eastern Europe'))[T.South Asia]                   -11.2646      2.892     -3.894      0.000     -16.973      -5.556
C(region, Treatment(reference='Eastern Europe'))[T.South-East Asia]              -10.6508      2.474     -4.305      0.000     -15.534      -5.767
C(region, Treatment(reference='Eastern Europe'))[T.Sub-Saharan Africa]           -11.5257      2.143     -5.378      0.000     -15.755      -7.296
C(region, Treatment(reference='Eastern Europe'))[T.W. Europe & N. America]        11.5495      2.388      4.836      0.000       6.835      16.264
gdp_per_capita                                                                     0.0002   2.82e-05      5.596      0.000       0.000       0.000
population_growth                                                                 -2.8314      0.620     -4.565      0.000      -4.056      -1.607
==============================================================================
Omnibus:                       47.119   Durbin-Watson:                   2.049
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              159.689
Skew:                           0.964   Prob(JB):                     2.11e-35
Kurtosis:                       7.109   Cond. No.                     3.15e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.15e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [36]:
# check for multicollinearity again
# it's fine if the intercept has a VIF > 5

vif = [variance_inflation_factor(model.model.exog, i)
       for i in range(model.model.exog.shape[1])]
vif

[7.02298507920304,
 1.3945049982898694,
 1.1363250270619671,
 1.6716562608582881,
 1.9931486808552916,
 1.4363824277022874,
 1.414935827140085,
 1.3996894561322835,
 3.5125202010286034,
 2.9086066360723244,
 2.2820360214691555,
 1.9819192129136003]

While we have successfully mitigated multicollinearity, removing the interactions also decreased the goodness of fit. However, R^2 is still at an acceptable level.

**Important:** Keep in mind that we are again performing multiple hypothesis testing here. If you report p-values or significant coefficients, corrections need to be applied.

## Basic NLP Classifiers

In [7]:
import nltk

# Download required NLTK data (run once)
for pkg in ["twitter_samples", "stopwords"]:
    nltk.download(pkg, quiet=True)

In [17]:
from nltk.corpus import twitter_samples, stopwords

# Load data and build (text, label) pairs
pos = twitter_samples.strings("positive_tweets.json")
neg = twitter_samples.strings("negative_tweets.json")

texts  = pos + neg
labels = [1] * len(pos) + [0] * len(neg)   # 1 = positive, 0 = negative
texts[:10]

['#FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)',
 '@Lamb2ja Hey James! How odd :/ Please call our Contact Centre on 02392441234 and we will be able to assist you :) Many thanks!',
 '@DespiteOfficial we had a listen last night :) As You Bleed is an amazing track. When are you in Scotland?!',
 '@97sides CONGRATS :)',
 'yeaaaah yippppy!!!  my accnt verified rqst has succeed got a blue tick mark on my fb profile :) in 15 days',
 '@BhaktisBanter @PallaviRuhail This one is irresistible :)\n#FlipkartFashionFriday http://t.co/EbZ0L2VENM',
 "We don't like to keep our lovely customers waiting for long! We hope you enjoy! Happy Friday! - LWWF :) https://t.co/smyYriipxI",
 '@Impatientraider On second thought, there’s just not enough time for a DD :) But new shorts entering system. Sheep must be buying.',
 'Jgh , but we have to go to Bayan :D bye',
 'As an act of mischievousness, am calling the ETL layer of our in-house warehousing 

**Important:** For Bag-of-Words classifiers, we need to apply some quite advanced preprocessing and heavily standardize our "words"

This is not required for more advanced classifiers that use LLMs (see next exercise)

In [16]:
from nltk.tokenize import TweetTokenizer
from nltk.stem import PorterStemmer
import regex as re

# Preprocessing: strip URLs/handles/hashtags, tokenize, lowercase, drop stopwords & punctuation, stem.
tokenizer  = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
stemmer    = PorterStemmer()
stop_words = set(stopwords.words("english"))

def preprocess(tweet: str) -> str:
    tweet = re.sub(r"https?://\S+|www\.\S+", "", tweet)   # URLs
    tweet = re.sub(r"#", "", tweet)                       # keep hashtag text, drop '#'
    tweet = re.sub(r"\$\w*|^RT[\s]+", "", tweet)          # tickers, retweet marker
    tokens = tokenizer.tokenize(tweet)
    cleaned = [
        stemmer.stem(tok)
        for tok in tokens
        if tok not in stop_words and tok.isalpha()
    ]
    return " ".join(cleaned)

cleaned_texts = [preprocess(t) for t in texts]
cleaned_texts[:10]

['followfriday top engag member commun week',
 'hey jame odd pleas call contact centr abl assist mani thank',
 'listen last night bleed amaz track scotland',
 'congrat',
 'yeaaah yipppi accnt verifi rqst succeed got blue tick mark fb profil day',
 'one irresist flipkartfashionfriday',
 'like keep love custom wait long hope enjoy happi friday lwwf',
 'second thought enough time dd new short enter system sheep must buy',
 'jgh go bayan bye',
 'act mischiev call etl layer wareh app katamari well name impli']

In [18]:
from sklearn.model_selection import train_test_split

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    cleaned_texts, labels, test_size=0.2, random_state=42, stratify=labels
)

In [19]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier

# Bag-of-Words vectorizer (fit on train only to avoid leakage)
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow  = vectorizer.transform(X_test)

# 6. Random Forest
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
)
clf.fit(X_train_bow, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [25]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Evaluate by predicting labels for the test set
y_pred = clf.predict(X_test_bow)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred, target_names=["neg", "pos"]))

Accuracy: 0.7135

Confusion matrix:
 [[743 257]
 [316 684]]

Report:
               precision    recall  f1-score   support

         neg       0.70      0.74      0.72      1000
         pos       0.73      0.68      0.70      1000

    accuracy                           0.71      2000
   macro avg       0.71      0.71      0.71      2000
weighted avg       0.71      0.71      0.71      2000



In [35]:
import numpy as np

# Add a baseline of randomly guessing labels for comparison
rng = np.random.default_rng()
y_rand = rng.integers(low=0, high=2, size=len(y_pred)) # how many classes are there in your application?
print("Random accuracy:", accuracy_score(y_test, y_rand))
print("\nRandom confusion matrix:\n", confusion_matrix(y_test, y_rand))
print("\nReport for random guessing:\n", classification_report(y_test, y_rand, target_names=["neg", "pos"]))

Random accuracy: 0.5085

Random confusion matrix:
 [[510 490]
 [493 507]]

Report for random guessing:
               precision    recall  f1-score   support

         neg       0.51      0.51      0.51      1000
         pos       0.51      0.51      0.51      1000

    accuracy                           0.51      2000
   macro avg       0.51      0.51      0.51      2000
weighted avg       0.51      0.51      0.51      2000



#### Once you are satisfied with the results, you can deploy your classifier on data that you hadn't labeled so far